# Milestone 4B: Fair Lending Analysis

Measures whether the production model (constrained + calibrated LightGBM at τ = 0.168, from 4A) produces fair outcomes across gender and age at the chosen operating point.

## Framework

Three fairness criteria are examined:

- **Demographic parity** -- approval rate equal across groups; the four-fifths rule (approval rate ratio ≥ 0.80) is the standard regulatory test.
- **Equalized odds** -- FN and FP rates equal across groups; measures whether the model makes the same kinds of mistakes at the same rates.
- **Predictive parity** -- actual default rate equal across groups conditional on predicted PD; measures whether probabilities mean the same thing for everyone.

These criteria are in tension when base default rates differ across groups (as they do here). No single model can satisfy all three simultaneously. This analysis reports on all three and discusses the tradeoffs.

## Scope

- Protected class proxies available: **CODE_GENDER**, **DAYS_BIRTH** (age).
- Not available: race, ethnicity, national origin. A real US fair lending audit would include these; this analysis is explicit about what it cannot measure.
- Primary threshold: τ = 0.168 (from 4A, FN:FP = 5:1).
- Sensitivity threshold: τ = 0.109 (FN:FP = 8:1) to show how disparities shift with stricter thresholds.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
from pathlib import Path

from src.data import load_joined, basic_clean
from src.splits import get_splits
from src.features import prepare_features, get_feature_columns

# Load the raw features (need gender, age for stratification)
df = load_joined()
df = basic_clean(df)
train_idx, val_idx, test_idx = get_splits(df)

df_features = prepare_features(df, include_ext_source_1=True)
val = df_features.loc[val_idx]

# Load predictions from Milestone 3
with open(Path("../data/processed/predictions.pkl"), "rb") as f:
    preds = pickle.load(f)
y_val = preds["y_val"]
y_val_pred = preds["y_val_pred_final"]

# Load threshold metadata from 4A
with open(Path("../data/processed/threshold_metadata.pkl"), "rb") as f:
    threshold_meta = pickle.load(f)
tau_primary = threshold_meta["chosen_threshold"]
tau_secondary = threshold_meta["secondary_threshold_8to1"]

print(f"Primary threshold: τ = {tau_primary:.4f} (from 4A, FN:FP = 5:1)")
print(f"Secondary threshold: τ = {tau_secondary:.4f} (FN:FP = 8:1)")

# Attach gender and age to a working DataFrame aligned with predictions
work = pd.DataFrame({
    "y_true": y_val,
    "y_pred": y_val_pred,
    "CODE_GENDER": val["CODE_GENDER"].values,
    "age_years": val["age_years"].values,
})
# Recreate the fair-lending age bands from EDA
work["age_band"] = pd.cut(
    work["age_years"],
    bins=[20, 25, 35, 45, 55, 62, 70],
    labels=["20-25", "25-35", "35-45", "45-55", "55-62", "62+"],
)

print(f"\nValidation set: {len(work):,} applicants")
print(f"Overall default rate: {work['y_true'].mean():.4f}")

In [ ]:
def group_metrics(work_df, threshold, group_col):
    """
    Compute approval rate, FN rate, FP rate, and observed default rate for each group.

    - approval_rate: fraction of applicants approved (y_pred <= threshold)
    - fn_rate: FN / total defaulters = fraction of actual defaulters who got approved.
      This is 1 - recall on the positive class.
    - fp_rate: FP / total non-defaulters = fraction of actual non-defaulters who got declined.
      This is the false positive rate.
    - default_rate: raw default rate in the group (from the data, not the model)
    """
    results = []
    for group, subgroup in work_df.dropna(subset=[group_col]).groupby(group_col, observed=True):
        declined = subgroup["y_pred"] > threshold
        approved = ~declined
        actual_default = subgroup["y_true"] == 1
        actual_nondefault = subgroup["y_true"] == 0

        n = len(subgroup)
        approval_rate = approved.mean()
        # FN rate: among actual defaulters, what fraction did we approve?
        fn_rate = (approved & actual_default).sum() / max(actual_default.sum(), 1)
        # FP rate: among actual non-defaulters, what fraction did we decline?
        fp_rate = (declined & actual_nondefault).sum() / max(actual_nondefault.sum(), 1)
        default_rate = actual_default.mean()

        results.append({
            group_col: group,
            "n": n,
            "approval_rate": approval_rate,
            "fn_rate": fn_rate,
            "fp_rate": fp_rate,
            "default_rate": default_rate,
        })
    return pd.DataFrame(results)


def four_fifths_test(metrics_df, reference_group_col):
    """
    Compute the approval rate ratio relative to the group with the highest approval rate.
    Ratio < 0.80 triggers the four-fifths rule presumption.
    """
    max_approval = metrics_df["approval_rate"].max()
    metrics_df = metrics_df.copy()
    metrics_df["approval_rate_ratio"] = metrics_df["approval_rate"] / max_approval
    metrics_df["passes_four_fifths"] = metrics_df["approval_rate_ratio"] >= 0.80
    return metrics_df

In [ ]:
# Gender analysis at τ = 0.168
gender_primary = group_metrics(work, tau_primary, "CODE_GENDER")
gender_primary = four_fifths_test(gender_primary, "CODE_GENDER")

print(f"Gender disparity at τ = {tau_primary:.4f}:\n")
print(gender_primary.round(4).to_string(index=False))

In [ ]:
# Age analysis at τ = 0.168
age_primary = group_metrics(work, tau_primary, "age_band")
age_primary = four_fifths_test(age_primary, "age_band")

print(f"Age disparity at τ = {tau_primary:.4f}:\n")
print(age_primary.round(4).to_string(index=False))

In [ ]:
# Rerun both at the secondary threshold (more conservative)
gender_secondary = four_fifths_test(group_metrics(work, tau_secondary, "CODE_GENDER"), "CODE_GENDER")
age_secondary = four_fifths_test(group_metrics(work, tau_secondary, "age_band"), "age_band")

print(f"Gender disparity at τ = {tau_secondary:.4f} (stricter threshold):\n")
print(gender_secondary.round(4).to_string(index=False))
print()
print(f"Age disparity at τ = {tau_secondary:.4f}:\n")
print(age_secondary.round(4).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Gender approval rate at both thresholds
for i, (df_r, tau, label) in enumerate([
    (gender_primary, tau_primary, "τ = 0.168 (FN:FP = 5:1)"),
    (gender_secondary, tau_secondary, "τ = 0.109 (FN:FP = 8:1)"),
]):
    ax = axes[0, i]
    bars = ax.bar(df_r["CODE_GENDER"], df_r["approval_rate"], color=["steelblue", "indianred"])
    ax.axhline(df_r["approval_rate"].max() * 0.80, color="black", linestyle="--",
               alpha=0.6, label="Four-fifths floor")
    ax.set_ylim(0, 1.0)
    ax.set_ylabel("Approval rate")
    ax.set_title(f"Gender: approval rate at {label}")
    ax.legend()
    for bar, ratio in zip(bars, df_r["approval_rate_ratio"]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                f"ratio: {ratio:.3f}", ha="center", fontsize=9)

# Age approval rate at both thresholds
for i, (df_r, tau, label) in enumerate([
    (age_primary, tau_primary, "τ = 0.168 (FN:FP = 5:1)"),
    (age_secondary, tau_secondary, "τ = 0.109 (FN:FP = 8:1)"),
]):
    ax = axes[1, i]
    max_approval = df_r["approval_rate"].max()
    colors = ["indianred" if r < 0.80 else "steelblue" for r in df_r["approval_rate_ratio"]]
    bars = ax.bar(df_r["age_band"].astype(str), df_r["approval_rate"], color=colors)
    ax.axhline(max_approval * 0.80, color="black", linestyle="--",
               alpha=0.6, label="Four-fifths floor")
    ax.set_ylim(0, 1.0)
    ax.set_ylabel("Approval rate")
    ax.set_title(f"Age: approval rate at {label}")
    ax.legend()
    for bar, ratio in zip(bars, df_r["approval_rate_ratio"]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                f"{ratio:.2f}", ha="center", fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Equalized odds: compare FN rate and FP rate across groups at primary threshold
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gender error rates
gender_err = gender_primary[["CODE_GENDER", "fn_rate", "fp_rate"]].copy()
gender_err_long = gender_err.melt(id_vars="CODE_GENDER", var_name="error_type", value_name="rate")

x = np.arange(len(gender_err))
width = 0.35
axes[0].bar(x - width/2, gender_err["fn_rate"], width, label="FN rate (approved defaulters)", color="steelblue")
axes[0].bar(x + width/2, gender_err["fp_rate"], width, label="FP rate (declined non-defaulters)", color="indianred")
axes[0].set_xticks(x)
axes[0].set_xticklabels(gender_err["CODE_GENDER"])
axes[0].set_ylabel("Error rate")
axes[0].set_title(f"Gender: error rates at τ = {tau_primary:.4f}")
axes[0].legend()
axes[0].grid(alpha=0.3, axis="y")

# Age error rates
age_err = age_primary[["age_band", "fn_rate", "fp_rate"]].copy()
x = np.arange(len(age_err))
axes[1].bar(x - width/2, age_err["fn_rate"], width, label="FN rate", color="steelblue")
axes[1].bar(x + width/2, age_err["fp_rate"], width, label="FP rate", color="indianred")
axes[1].set_xticks(x)
axes[1].set_xticklabels(age_err["age_band"].astype(str))
axes[1].set_ylabel("Error rate")
axes[1].set_title(f"Age: error rates at τ = {tau_primary:.4f}")
axes[1].legend()
axes[1].grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

In [ ]:
# How much of the observed disparity comes from the model vs from the data?
# Compare: raw default rate gap (data) vs approval rate gap (model output)

print("Gender: how much disparity comes from data vs model\n")
gender_diagnostic = gender_primary[["CODE_GENDER", "n", "default_rate", "approval_rate", "approval_rate_ratio"]].copy()
print(gender_diagnostic.round(4).to_string(index=False))

# Compute the "expected" approval rate ratio if the model was perfectly calibrated on default
# and made no other distinction. A group with 2x default rate should have ~2x decline rate.
print()
print("Age: how much disparity comes from data vs model\n")
age_diagnostic = age_primary[["age_band", "n", "default_rate", "approval_rate", "approval_rate_ratio"]].copy()
print(age_diagnostic.round(4).to_string(index=False))

In [ ]:
# Save the disparity results for use in 4B(ii) EXT_SOURCE_1 sensitivity analysis
disparity_results = {
    "gender_primary": gender_primary,
    "gender_secondary": gender_secondary,
    "age_primary": age_primary,
    "age_secondary": age_secondary,
    "tau_primary": tau_primary,
    "tau_secondary": tau_secondary,
}
with open(Path("../data/processed/disparity_baseline.pkl"), "wb") as f:
    pickle.dump(disparity_results, f)

print("Saved baseline disparity results for 4B(ii) sensitivity comparison.")

**Section 4B takeaways:**

**Four-fifths rule results at primary threshold (τ = 0.168):**

- Gender: **passes** (M ratio = 0.906 vs F, four-fifths floor = 0.80)
- Age: **fails** -- the 20-25 band has approval rate ratio 0.667 vs the 62+ reference. All other age bands pass (ratio ≥ 0.807).

**Four-fifths rule results at secondary threshold (τ = 0.109):**

- Gender: **passes** but tighter (M ratio = 0.828)
- Age: **fails more broadly** -- both 20-25 (ratio 0.466) and 25-35 (ratio 0.676) fall below the four-fifths floor.

**Error rate parity:**

- Gender FN rate gap: 15.5 percentage points (F 68.6% vs M 53.0%)
- Gender FP rate gap: 6.7 percentage points (F 7.8% vs M 14.6%)
- Age FN rate gap: 53.5 percentage points across age bands (20-25: 36.0% → 62+: 89.5%)
- Age FP rate gap: 29 percentage points (20-25: 30.8% → 62+: 1.8%)

The model catches younger defaulters more aggressively (low FN rate) at the cost of also declining many young non-defaulters (high FP rate). The pattern reverses for older applicants: FP rate drops to near zero, but FN rate climbs above 89% -- the model rarely declines older applicants, even those who go on to default.

**Interpretation:**

- **Gender is not the fair-lending problem here.** The 1.54x default-rate gap between men and women translates to only a 0.906 approval-rate ratio, well above the four-fifths floor. The model is not amplifying the raw group difference.
- **Age is the fair-lending problem.** At the primary threshold, the 20-25 band has a 33% shortfall in approval rate vs the reference band, triggering four-fifths presumption. This is not driven by direct use of age alone -- it reflects the model's use of features that correlate with age (employment tenure, EXT_SOURCE scores, occupation, bureau history).
- **The model is compressing, not amplifying, group-level differences.** The 2.77x default-rate ratio between the 20-25 and 62+ bands is only a 1.5x approval-rate ratio. Age still fails the test because the raw disparity is so large that even a compressed version violates four-fifths.
- **Stricter thresholds make fair lending worse.** Moving from τ = 0.168 to τ = 0.109 pushes a second age band (25-35) below the four-fifths floor. The threshold choice from 4A is not just an economic decision -- it's a fair lending decision.

**What this means:**

- The model is not deployment-ready as-is at either threshold. The 20-25 age band's approval rate ratio of 0.667 would almost certainly trigger regulatory review in a US context.
- The tradeoff is stark: the 5:1 cost-optimal threshold produces a fair lending failure on age; a stricter 8:1 threshold makes it worse; a looser threshold would need FN:FP < 3.5 (approval rate ≈ 94%) to have any chance of passing four-fifths on age.
- Feature-level intervention is probably needed. Candidate approaches worth exploring: (a) drop or constrain EXT_SOURCE_1, which EDA showed embeds heavy age structure; (b) apply post-processing to equalize approval rates by age band; (c) accept a different fairness criterion (e.g., equalized odds) rather than demographic parity.

**Limitation:** This analysis uses gender and age only. Race, ethnicity, and national origin are not in the Home Credit dataset. A production fair lending audit in the US would include these; this analysis is explicit about its scope.

**Follow-up (Section 4B(ii)):** retrain the model without EXT_SOURCE_1 (the demographically-loaded external score identified in EDA) and measure the fair lending vs performance tradeoff. This is the direct test of whether one specific feature can be dropped to improve fair lending outcomes.

## Section 4B(ii): EXT_SOURCE_1 Sensitivity Analysis

Tests whether removing EXT_SOURCE_1 — the external credit score with heavy demographic structure identified in EDA (women 0.546 vs men 0.407; 20-25 band 0.28 vs 62+ band 0.75) — improves fair lending outcomes.

**Method:** Retrain the full pipeline (LightGBM with monotonic constraints, isotonic calibration) on the feature set excluding EXT_SOURCE_1. Re-optimize the threshold at FN:FP = 5:1 (matching 4A). Rerun the fair lending analysis. Compare to the baseline.

**What we're measuring:**
- Cost: how much AUC / KS / Brier do we lose without EXT_SOURCE_1?
- Benefit: do gender and age approval rate ratios improve?
- Verdict: is EXT_SOURCE_1 the driver of the fair lending failure, or is the disparity structural (not attributable to any single feature)?

In [ ]:
# Additional imports needed for retraining the ablation model
import lightgbm as lgb
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
from scipy.stats import ks_2samp


def ks_statistic(y_true, y_pred):
    return ks_2samp(y_pred[y_true == 1], y_pred[y_true == 0]).statistic


# Prepare features WITHOUT EXT_SOURCE_1
df_features_ablation = prepare_features(df, include_ext_source_1=False)
train_abl = df_features_ablation.loc[train_idx]
val_abl = df_features_ablation.loc[val_idx]
test_abl = df_features_ablation.loc[test_idx]

feature_cols_abl = get_feature_columns(include_ext_source_1=False)
categorical_cols = [
    "NAME_CONTRACT_TYPE", "CODE_GENDER", "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS", "NAME_HOUSING_TYPE", "OCCUPATION_TYPE",
]

X_train_abl, y_train = train_abl[feature_cols_abl], train_abl["TARGET"]
X_val_abl, y_val = val_abl[feature_cols_abl], val_abl["TARGET"]
X_test_abl, y_test = test_abl[feature_cols_abl], test_abl["TARGET"]

print(f"Ablation feature set: {len(feature_cols_abl)} features (baseline was 40; dropped EXT_SOURCE_1)")

In [ ]:
# Same monotonic constraints as the baseline, but skip EXT_SOURCE_1
constraint_directions = {
    "EXT_SOURCE_2": -1,
    "EXT_SOURCE_3": -1,
    "bureau_overdue_max": +1,
    "employment_years": -1,
    "age_years": -1,
    "payment_to_income": +1,
    "loan_to_income": +1,
}
monotone_constraints_abl = [
    constraint_directions.get(col, 0) for col in feature_cols_abl
]

# Convert categoricals
X_train_abl_lgb = X_train_abl.copy()
X_val_abl_lgb = X_val_abl.copy()
X_test_abl_lgb = X_test_abl.copy()
for col in categorical_cols:
    X_train_abl_lgb[col] = X_train_abl_lgb[col].astype("category")
    X_val_abl_lgb[col] = X_val_abl_lgb[col].astype("category")
    X_test_abl_lgb[col] = X_test_abl_lgb[col].astype("category")

# Same params as the baseline
lgb_params_abl = {
    "objective": "binary", "metric": "auc",
    "learning_rate": 0.05, "num_leaves": 63, "max_depth": -1,
    "min_data_in_leaf": 100, "feature_fraction": 0.8,
    "bagging_fraction": 0.8, "bagging_freq": 5,
    "random_state": 42, "verbose": -1,
    "monotone_constraints": monotone_constraints_abl,
    "monotone_constraints_method": "intermediate",
}

lgb_train_ds = lgb.Dataset(X_train_abl_lgb, label=y_train, categorical_feature=categorical_cols)
lgb_val_ds = lgb.Dataset(X_val_abl_lgb, label=y_val, categorical_feature=categorical_cols, reference=lgb_train_ds)

lgb_model_abl = lgb.train(
    lgb_params_abl,
    lgb_train_ds,
    num_boost_round=2000,
    valid_sets=[lgb_train_ds, lgb_val_ds],
    valid_names=["train", "val"],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=0),
    ],
)

print(f"Ablation model best iteration: {lgb_model_abl.best_iteration}")

In [ ]:
# Predict on val and test
y_val_pred_abl_raw = lgb_model_abl.predict(X_val_abl_lgb, num_iteration=lgb_model_abl.best_iteration)
y_test_pred_abl_raw = lgb_model_abl.predict(X_test_abl_lgb, num_iteration=lgb_model_abl.best_iteration)

# Isotonic calibration on val
iso_abl = IsotonicRegression(out_of_bounds="clip")
iso_abl.fit(y_val_pred_abl_raw, y_val.values)
y_val_pred_abl = iso_abl.predict(y_val_pred_abl_raw)
y_test_pred_abl = iso_abl.predict(y_test_pred_abl_raw)

# Compare metrics vs baseline
# Load baseline predictions
with open(Path("../data/processed/predictions.pkl"), "rb") as f:
    baseline_preds = pickle.load(f)
y_val_pred_baseline = baseline_preds["y_val_pred_final"]

print("Performance comparison (validation set):")
print(f"{'Metric':<8} {'Baseline':>12} {'Ablation':>12} {'Δ':>10}")

for metric_name, metric_fn in [
    ("AUC", lambda y, p: roc_auc_score(y, p)),
    ("KS", lambda y, p: ks_statistic(y, p)),
    ("Brier", lambda y, p: brier_score_loss(y, p)),
]:
    base = metric_fn(y_val.values, y_val_pred_baseline)
    abl = metric_fn(y_val.values, y_val_pred_abl)
    print(f"{metric_name:<8} {base:>12.4f} {abl:>12.4f} {abl - base:>+10.4f}")

In [ ]:
# Reuse the cost logic from 4A: sweep thresholds, find cost minimum for FN:FP = 5:1
COST_FP = 80_000
FN_FP_RATIO = 5.0
FN_COST = COST_FP * FN_FP_RATIO

thresholds_sweep = np.linspace(0.01, 0.50, 100)
costs = []
for t in thresholds_sweep:
    declined = y_val_pred_abl > t
    fn = ((~declined) & (y_val.values == 1)).sum()
    fp = (declined & (y_val.values == 0)).sum()
    costs.append(fn * FN_COST + fp * COST_FP)

opt_idx = np.argmin(costs)
tau_abl_primary = thresholds_sweep[opt_idx]
approval_abl = (y_val_pred_abl <= tau_abl_primary).mean()

# Baseline threshold for reference
with open(Path("../data/processed/threshold_metadata.pkl"), "rb") as f:
    threshold_meta = pickle.load(f)
tau_baseline = threshold_meta["chosen_threshold"]
approval_baseline = threshold_meta["chosen_approval_rate"]

print(f"Threshold comparison at FN:FP = 5:1:")
print(f"  Baseline (with EXT_SOURCE_1):    τ = {tau_baseline:.4f}, approval = {approval_baseline:.2%}")
print(f"  Ablation (no EXT_SOURCE_1):      τ = {tau_abl_primary:.4f}, approval = {approval_abl:.2%}")

In [ ]:
# Attach demographics to the ablation predictions
work_abl = pd.DataFrame({
    "y_true": y_val.values,
    "y_pred": y_val_pred_abl,
    "CODE_GENDER": val_abl["CODE_GENDER"].values,
    "age_years": val_abl["age_years"].values,
})
work_abl["age_band"] = pd.cut(
    work_abl["age_years"],
    bins=[20, 25, 35, 45, 55, 62, 70],
    labels=["20-25", "25-35", "35-45", "45-55", "55-62", "62+"],
)

# Fair lending metrics at the ablation model's re-optimized threshold
# Uses group_metrics and four_fifths_test defined in cell 2
gender_abl = four_fifths_test(group_metrics(work_abl, tau_abl_primary, "CODE_GENDER"), "CODE_GENDER")
age_abl = four_fifths_test(group_metrics(work_abl, tau_abl_primary, "age_band"), "age_band")

print(f"Ablation model gender disparity at τ = {tau_abl_primary:.4f}:\n")
print(gender_abl.round(4).to_string(index=False))
print()
print(f"Ablation model age disparity at τ = {tau_abl_primary:.4f}:\n")
print(age_abl.round(4).to_string(index=False))

In [ ]:
# Load baseline disparity results
with open(Path("../data/processed/disparity_baseline.pkl"), "rb") as f:
    baseline_disparity = pickle.load(f)

gender_base = baseline_disparity["gender_primary"]
age_base = baseline_disparity["age_primary"]

# Side-by-side comparison
print("=" * 70)
print("GENDER — approval rate ratios (baseline vs ablation)")
print("=" * 70)
gender_comp = pd.DataFrame({
    "CODE_GENDER": gender_base["CODE_GENDER"].values,
    "baseline_approval": gender_base["approval_rate"].values,
    "baseline_ratio": gender_base["approval_rate_ratio"].values,
    "ablation_approval": gender_abl["approval_rate"].values,
    "ablation_ratio": gender_abl["approval_rate_ratio"].values,
})
gender_comp["ratio_change"] = gender_comp["ablation_ratio"] - gender_comp["baseline_ratio"]
print(gender_comp.round(4).to_string(index=False))
print()

print("=" * 70)
print("AGE — approval rate ratios (baseline vs ablation)")
print("=" * 70)
age_comp = pd.DataFrame({
    "age_band": age_base["age_band"].astype(str).values,
    "baseline_approval": age_base["approval_rate"].values,
    "baseline_ratio": age_base["approval_rate_ratio"].values,
    "ablation_approval": age_abl["approval_rate"].values,
    "ablation_ratio": age_abl["approval_rate_ratio"].values,
})
age_comp["ratio_change"] = age_comp["ablation_ratio"] - age_comp["baseline_ratio"]
print(age_comp.round(4).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gender comparison
x = np.arange(len(gender_comp))
width = 0.35
axes[0].bar(x - width/2, gender_comp["baseline_ratio"], width, label="With EXT_SOURCE_1", color="steelblue")
axes[0].bar(x + width/2, gender_comp["ablation_ratio"], width, label="Without EXT_SOURCE_1", color="darkorange")
axes[0].axhline(0.80, color="black", linestyle="--", alpha=0.5, label="Four-fifths floor")
axes[0].set_xticks(x)
axes[0].set_xticklabels(gender_comp["CODE_GENDER"])
axes[0].set_ylabel("Approval rate ratio")
axes[0].set_title("Gender: EXT_SOURCE_1 ablation effect")
axes[0].set_ylim(0, 1.1)
axes[0].legend()
axes[0].grid(alpha=0.3, axis="y")

# Age comparison
x = np.arange(len(age_comp))
axes[1].bar(x - width/2, age_comp["baseline_ratio"], width, label="With EXT_SOURCE_1", color="steelblue")
axes[1].bar(x + width/2, age_comp["ablation_ratio"], width, label="Without EXT_SOURCE_1", color="darkorange")
axes[1].axhline(0.80, color="black", linestyle="--", alpha=0.5, label="Four-fifths floor")
axes[1].set_xticks(x)
axes[1].set_xticklabels(age_comp["age_band"].astype(str))
axes[1].set_ylabel("Approval rate ratio")
axes[1].set_title("Age: EXT_SOURCE_1 ablation effect")
axes[1].set_ylim(0, 1.1)
axes[1].legend()
axes[1].grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

In [ ]:
# Save the ablation model results for downstream reference
ablation_results = {
    "auc_baseline": 0.7626,
    "auc_ablation": 0.7567,
    "ks_baseline": 0.3911,
    "ks_ablation": 0.3786,
    "gender_ablation": gender_abl,
    "age_ablation": age_abl,
    "tau_ablation": tau_abl_primary,
}
with open(Path("../data/processed/ablation_results.pkl"), "wb") as f:
    pickle.dump(ablation_results, f)

print("Saved 4B(ii) ablation results.")

**Section 4B(ii) takeaways:**

**Performance cost of dropping EXT_SOURCE_1:**
- AUC: 0.7626 → 0.7567 (Δ = −0.0059)
- KS: 0.3911 → 0.3786 (Δ = −0.0125)
- Brier: 0.0675 → 0.0679 (Δ = +0.0004; calibration essentially unchanged)

**Fair lending gain:**
- Gender approval ratio (M vs F): 0.906 → 0.914 (small)
- Age 20-25 approval ratio (vs 62+): 0.667 → 0.704 (+0.037; meaningful but still fails four-fifths)
- Ablation model still fails four-fifths on the 20-25 band at τ = 0.168.

**Verdict: EXT_SOURCE_1 was contributing to the age disparity, but not driving it.**

The EDA hypothesis -- that EXT_SOURCE_1 carried demographic structure worth removing -- was directionally confirmed. Dropping it moved the age 20-25 approval ratio from 0.667 to 0.704 and the 25-35 band from a borderline pass to a comfortable pass. But the primary fair lending failure (20-25 vs 62+) persists. The disparity is structural: it reflects the joint effect of many features (employment tenure, occupation, EXT_SOURCE_2/3, bureau history) that all correlate with both age and default risk.

**What this tells us:**

- The AUC cost of dropping EXT_SOURCE_1 is genuinely small (0.006). A credit officer weighing "small AUC cost for meaningful fair lending improvement" would likely take the trade even without fully fixing the failure.
- Some of the observed improvement comes from the ablation model's slightly compressed PD distribution -- at fixed τ = 0.168, the ablation model approves 89.3% vs 87.7% baseline. Some of the disparity improvement is because more people are approved overall, not purely because approvals are more equal.
- Fixing the four-fifths failure on age would require either (a) a substantially higher operating threshold, contradicting cost optimization; (b) post-processing to equalize approval rates by protected group; or (c) accepting equalized odds (or another fairness criterion) rather than demographic parity.

**Practical recommendation:** if this model went into production, drop EXT_SOURCE_1 (small cost, real improvement, better regulatory story) and combine with either a post-processing step or a group-aware threshold policy to reach four-fifths compliance on age. This is the kind of decision that would be escalated to the fair lending compliance officer in a real credit risk team.

**Broader implication:** this analysis illustrates a fundamental tension in credit modeling: when protected-class proxies correlate with default risk (as age does in this dataset, with a 2.77x default rate gap between 20-25 and 62+), no single feature-level intervention will eliminate group-level outcome differences. Fair lending compliance requires accepting either (a) a business tradeoff on cost-optimal thresholds, or (b) explicit fairness-aware post-processing, or (c) rejecting demographic parity as the correct fairness criterion for the domain.